# AGAR-RL: Autonomous Multi-Agent Deep Reinforcement Learning Pipeline

Ce notebook permet d'exécuter l'entraînement distribué par Deep Reinforcement Learning (PPO & Self-Play Multi-Modèles) directement sur **Google Colab** (GPU T4/A100) ou en local.

## 1. Détection de l'Environnement et Installation des Dépendances

In [ ]:
import os, sys

# 1. Récupération des dernières modifications ou clonage
if os.path.exists('.git'):
    print('🔄 Récupération des dernières mises à jour du repo...')
    !git pull origin main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et mise à jour...')
    %cd agario
    !git pull origin main
else:
    print('🌐 Environnement distant Colab détecté. Clonage du repo...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Configuration du PYTHONPATH et installation des dépendances
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip install -q -r requirements.txt tensorboard

# 3. Vérification GPU CUDA
import torch
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU actif : {torch.cuda.get_device_name(0)}')
else:
    print('Exécution sur CPU.')


## 2. Validation de la Suite de Tests (24 Tests)

In [ ]:
!python -m pytest -v

## 3. Monitoring TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/tensorboard

## 4. Démarrer l'Entraînement Multi-Agent (Self-Play & X Modèles dans l'Arène)

In [ ]:
# Lance 16 arènes en parallèle (10 adversaires par arène = 176 joueurs en compétition simultanée)
# Le pool de Self-Play s'actualise toutes les 50 000 étapes pour injecter les nouveaux modèles dans l'arène
!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 1000000 \
    --batch-size 128 \
    --n-steps 2048 \
    --pool-interval 50000 \
    --device auto

## 5. Visualiser & Enregistrer les Parties en Vidéo HD (Replay avec Vecteurs de Décision)

In [ ]:
# 1. Enregistrement d'une partie HD avec le modèle entraîné
# Affiche la caméra fluide, les flèches de décision (fuite/nourriture/split) et le HUD de masse
!python src/inference/record_match.py \
    --model checkpoints/ppo/ppo_latest.zip \
    --output recordings/eval_match.mp4 \
    --steps 800

# 2. Visualisation directe du Replay Vidéo dans le Notebook
import os
from IPython.display import HTML, display
from base64 import b64encode

video_path = 'recordings/eval_match.mp4'
if os.path.exists(video_path):
    mp4_bytes = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="800" height="450" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
else:
    print('Vidéo non trouvée.')


## 6. Exporter la Politique vers ONNX (< 0.02 ms de latence CPU)

In [ ]:
!python src/inference/export_onnx.py \
    --model checkpoints/ppo/ppo_final.zip \
    --output models/model.onnx

## 7. Téléchargement du Modèle Entraîné et de la Vidéo

In [ ]:
try:
    from google.colab import files
    files.download('models/model.onnx')
    files.download('checkpoints/ppo/ppo_final.zip')
    print('Téléchargement Colab initié.')
except ImportError:
    print('Fichiers sauvegardés localement dans models/ et checkpoints/.')